# ACT DR6 Gmu-P grid

Fig. 4-style ACT DR6 curl constraint in the `(Gmu, P)` plane. This uses the bundled ACT DR6 MV curl bandpowers with a diagonal Gaussian bandpower likelihood. Because the current ACT table has band centers but not bandpower window functions, the theory is evaluated at the ACT effective multipoles.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from tqdm import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "strings").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "serif"


In [ ]:
import strings as cs

sys.path.insert(0, str(PROJECT_ROOT / "fortran_backend"))
import cosmic_cl_backend as fcl

ACT = cs.ACTDR6
GREF = 1.0e-10

P_vals = np.logspace(-8, 0, 96)
Gmu_vals = np.logspace(-14, -3, 176)
levels = np.array([2.30, 6.18, 11.83])

theory_settings = dict(
    n_k=48,
    n_chi=1024,
    k_min=1.0e-4,
    k_max=1.0e-1,
)

PLOT_DIR = PROJECT_ROOT / "notebooks" / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
GRID_PATH = PLOT_DIR / "act_dr6_gmup_grid.npz"


In [ ]:
def compute_act_grid():
    ref_grid = np.empty((len(P_vals), len(ACT.L)))
    for i, P in enumerate(tqdm(P_vals, desc="ACT DR6 theory refs")):
        ref_grid[i] = fcl.compute_cl(GREF, P, ell_arr=ACT.L, **theory_settings)

    scale = (Gmu_vals / GREF) ** 2
    chi2_grid = np.empty((len(Gmu_vals), len(P_vals)))
    for i in range(len(P_vals)):
        model = scale[:, None] * ref_grid[i][None, :]
        chi2_grid[:, i] = np.sum(((ACT.CL[None, :] - model) / ACT.ER[None, :]) ** 2, axis=1)

    chi2_min = float(np.min(chi2_grid))
    delta_chi2 = chi2_grid - chi2_min
    return ref_grid, chi2_grid, delta_chi2, chi2_min


def cache_matches(cache):
    return (
        np.array_equal(cache["P_vals"], P_vals)
        and np.array_equal(cache["Gmu_vals"], Gmu_vals)
        and np.array_equal(cache["levels"], levels)
    )


if GRID_PATH.exists():
    cache = np.load(GRID_PATH)
    if cache_matches(cache):
        ref_grid = cache["ref_grid"]
        chi2_grid = cache["chi2_grid"]
        delta_chi2 = cache["delta_chi2"]
        chi2_min = float(cache["chi2_min"])
    else:
        ref_grid, chi2_grid, delta_chi2, chi2_min = compute_act_grid()
else:
    ref_grid, chi2_grid, delta_chi2, chi2_min = compute_act_grid()


In [ ]:
def upper_boundary(delta_chi2, level):
    log_g = np.log10(Gmu_vals)
    boundary = np.full(len(P_vals), np.nan)

    for i in range(len(P_vals)):
        d = delta_chi2[:, i]
        ok = d <= level
        if not np.any(ok):
            continue

        last = int(np.where(ok)[0].max())
        if last >= len(Gmu_vals) - 1:
            boundary[i] = Gmu_vals[-1]
            continue

        d0, d1 = d[last], d[last + 1]
        if d1 == d0:
            boundary[i] = Gmu_vals[last]
        else:
            t = (level - d0) / (d1 - d0)
            boundary[i] = 10 ** (log_g[last] + t * (log_g[last + 1] - log_g[last]))

    return boundary


boundary_1sigma = upper_boundary(delta_chi2, levels[0])
boundary_2sigma = upper_boundary(delta_chi2, levels[1])
boundary_3sigma = upper_boundary(delta_chi2, levels[2])

q_2sigma = float(np.nanmax(boundary_2sigma / P_vals))
gmu_p1_2sigma = float(10 ** np.interp(0.0, np.log10(P_vals), np.log10(boundary_2sigma)))
min_j, min_i = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)

np.savez(
    GRID_PATH,
    P_vals=P_vals,
    Gmu_vals=Gmu_vals,
    chi2_grid=chi2_grid,
    delta_chi2=delta_chi2,
    ref_grid=ref_grid,
    levels=levels,
    boundary_68=boundary_1sigma,
    boundary_95=boundary_2sigma,
    boundary_997=boundary_3sigma,
    q95=q_2sigma,
    gmu_p1_95=gmu_p1_2sigma,
    chi2_min=chi2_min,
    best_P=P_vals[min_i],
    best_Gmu=Gmu_vals[min_j],
)

{
    "chi2_min": chi2_min,
    "best_P": P_vals[min_i],
    "best_Gmu": Gmu_vals[min_j],
    "Gmu_over_P_2sigma": q_2sigma,
    "Gmu_P_equals_1_2sigma": gmu_p1_2sigma,
}


In [ ]:
Pgks, Ggks = np.loadtxt(PROJECT_ROOT / "data" / "gks.txt", unpack=True)
P_mesh, Gmu_mesh = np.meshgrid(P_vals, Gmu_vals)

fig, ax = plt.subplots(figsize=(6.4, 5.8))
colors = ["#c7e9c0", "#74c476", "#238b45"]
max_level = max(float(np.nanmax(delta_chi2)), 20.0)

ax.contourf(
    P_mesh,
    Gmu_mesh,
    delta_chi2,
    levels=[levels[0], levels[1], levels[2], max_level],
    colors=colors,
    alpha=0.82,
    extend="max",
)
ax.contour(
    P_mesh,
    Gmu_mesh,
    delta_chi2,
    levels=levels,
    colors=["#7fc97f", "#1b9e77", "#006d2c"],
    linewidths=[1.4, 1.8, 1.4],
)

ax.plot(P_vals, boundary_2sigma, color="black", lw=2.0)
ax.plot(Pgks, Ggks, color="black", ls="--", lw=1.4)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(P_vals[0], P_vals[-1])
ax.set_ylim(Gmu_vals[0], Gmu_vals[-1])
ax.set_xlabel("P")
ax.set_ylabel("Gmu")
ax.set_title("ACT DR6 curl constraint in the Gmu-P plane")

legend_handles = [
    Patch(facecolor=colors[0], edgecolor="none", alpha=0.82, label=r"$\Delta\chi^2>2.30$"),
    Patch(facecolor=colors[1], edgecolor="none", alpha=0.82, label=r"$\Delta\chi^2>6.18$"),
    Patch(facecolor=colors[2], edgecolor="none", alpha=0.82, label=r"$\Delta\chi^2>11.83$"),
    Line2D([0], [0], color="black", lw=2.0, label="2sigma upper boundary"),
    Line2D([0], [0], color="black", lw=1.4, ls="--", label="GKS"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9, frameon=True, framealpha=0.92)

ax.text(
    0.035,
    0.965,
    "ACT DR6 MV curl, diagonal bandpower likelihood\n"
    f"Gmu/P < {q_2sigma:.2e} (2sigma)\n"
    f"Gmu(P=1) < {gmu_p1_2sigma:.2e}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=9,
    bbox=dict(facecolor="white", edgecolor="0.75", boxstyle="round,pad=0.35", alpha=0.92),
)
ax.grid(True, which="both", alpha=0.18, lw=0.5)
fig.tight_layout()

png_path = PLOT_DIR / "act_dr6_gmup_final.png"
pdf_path = PLOT_DIR / "act_dr6_gmup_final.pdf"
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path)
plt.show()

print(f"saved {png_path}")
print(f"saved {pdf_path}")
